# 05쪽 · 조건에 따라 다른 경로 실행

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## 1. State 정의

이번 예제에서는 c와 d까지만 사용합니다.

In [ ]:
from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    a: int
    b: int
    c: NotRequired[int]
    d: NotRequired[int]

## 2. Node 정의

node2는 c의 두 배, node3는 c에서 2를 뺍니다.

In [ ]:
def node1(state: State) -> dict:
    return {"c": state["a"] + state["b"]}


def node2(state: State) -> dict:
    return {"d": state["c"] * 2}


def node3(state: State) -> dict:
    return {"d": state["c"] - 2}

## 3. 경로 선택 함수

c가 30 이상이면 large, 그보다 작으면 small입니다.

In [ ]:
def select_route(state):
    if state["c"] >= 30:
        return "large"
    else:
        return "small"

## 4. 그래프 연결

large는 node2, small은 node3로 연결합니다.

In [ ]:
builder = StateGraph(State)

builder.add_node("node1", node1)
builder.add_node("node2", node2)
builder.add_node("node3", node3)

builder.add_edge(START, "node1")
builder.add_conditional_edges(
    "node1",
    select_route,
    {
        "large": "node2",
        "small": "node3",
    },
)
builder.add_edge("node2", END)
builder.add_edge("node3", END)

graph = builder.compile()

## 5. 예시 A: 10과 20

large 경로를 따로 실행합니다.

In [ ]:
result1 = graph.invoke({
    "a": 10,
    "b": 20,
})

print(result1)

**결과 해설**

`c=30`, `d=60`입니다. node2를 실행합니다.

## 6. 예시 B: 2와 3

small 경로를 따로 실행합니다.

In [ ]:
result2 = graph.invoke({
    "a": 2,
    "b": 3,
})

print(result2)

**결과 해설**

`c=5`, `d=3`입니다. node3를 실행합니다.